**Notebook 02:**
# Preprocesamiento del Dataset CIC-IDS2017

Este notebook implementa el pipeline de preprocesamiento descrito en la Sección 3.4 de la metodología. Las acciones se ejecutan en el siguiente orden:

1. Limpieza de datos (NaN, infinitos, valores negativos inválidos)
2. Eliminación de columnas con varianza cero
3. Codificación binaria de la variable objetivo
4. Descarte semántico de columnas no numéricas
5. Partición estratificada Train/Test (80/20, semilla=42)
6. Filtro por correlación de Pearson (calculado solo en train)
7. Normalización con StandardScaler (ajustado solo en train)
8. Cálculo de ponderación de clases
9. Transformación secuencial para LSTM (ventana deslizante)
10. Guardado de artefactos para la fase de entrenamiento

---

## 1. Requisitos

```bash
pip install pandas numpy scikit-learn
```

Se requiere haber ejecutado previamente `01_exploracion_dataset.ipynb` para contar con el archivo `data/processed/CIC-IDS2017_unified.csv.gz`.

In [1]:
import os
import time
import pickle
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

print(f"Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Directorio de trabajo: {os.getcwd()}")

c:\Proyectos\TT-deteccionIntrusiones\tt_env\Lib\site-packages\sklearn\utils\_param_validation.py:11: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.5.4)
  from scipy.sparse import csr_matrix, issparse


Fecha de ejecución: 2026-10-19 15:58:10
Directorio de trabajo: c:\Proyectos\TT-deteccionIntrusiones


In [19]:
# ============================================================
# Configuración de rutas y parámetros
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TABLES_DIR = PROJECT_ROOT / "tables"
LOGS_DIR = PROJECT_ROOT / "logs"

for d in [PROCESSED_DIR, TABLES_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
CORR_THRESHOLD = 0.95
WINDOW_SIZE = 10

# Columnas con valores negativos inválidos detectados en el EDA
TEMPORAL_NEGATIVE_COLS = [
    "Flow Duration",
    "Flow IAT Mean",
    "Flow IAT Max",
    "Flow IAT Min",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Fwd IAT Min",
]

# Columnas de varianza cero detectadas en el EDA
ZERO_VAR_COLS = [
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate",
]

print(f"Semilla global: {RANDOM_STATE}")
print(f"Proporción de prueba: {TEST_SIZE}")
print(f"Umbral de correlación: {CORR_THRESHOLD}")
print(f"Tamaño de ventana LSTM: {WINDOW_SIZE}")
print(f"Columnas temporales a revisar: {len(TEMPORAL_NEGATIVE_COLS)}")
print(f"Columnas de varianza cero a eliminar: {len(ZERO_VAR_COLS)}")

Semilla global: 42
Proporción de prueba: 0.2
Umbral de correlación: 0.95
Tamaño de ventana LSTM: 10
Columnas temporales a revisar: 7
Columnas de varianza cero a eliminar: 8


In [3]:
# ============================================================
# Registro del pipeline
# ============================================================

pipeline_log = []

def log_step(step: str, rows_before: int, rows_after: int,
             cols_before: int, cols_after: int, detail: str = ""):
    """Registra cada paso del pipeline para trazabilidad."""
    entry = {
        "step": step,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_before - rows_after,
        "cols_before": cols_before,
        "cols_after": cols_after,
        "cols_removed": cols_before - cols_after,
        "detail": detail,
    }
    pipeline_log.append(entry)
    print(f"[{step}] Filas: {rows_before:,} -> {rows_after:,} "
          f"(-{rows_before - rows_after:,}) | "
          f"Columnas: {cols_before} -> {cols_after} "
          f"(-{cols_before - cols_after})")
    if detail:
        print(f"  Detalle: {detail}")

print("Función de registro cargada.")

Función de registro cargada.


## 2. Carga del dataset unificado

Se carga el archivo CSV comprimido generado en el notebook `01_exploracion_dataset.ipynb`.

In [4]:
# ============================================================
# Carga del dataset unificado
# ============================================================

input_path = PROCESSED_DIR / "CIC-IDS2017_unified.csv.gz"

if not input_path.exists():
    raise FileNotFoundError(
        f"No se encontró {input_path}. "
        "Ejecute primero 01_exploracion_dataset.ipynb."
    )

print(f"Cargando dataset desde: {input_path}")
t0 = time.time()

df = pd.read_csv(input_path, low_memory=False)
df.columns = df.columns.str.strip()

elapsed = time.time() - t0
print(f"Carga completada en {elapsed:.1f} segundos.")
print(f"Filas: {len(df):,}")
print(f"Columnas: {len(df.columns)}")
print(f"Uso de memoria: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

Cargando dataset desde: C:\Proyectos\TT-deteccionIntrusiones\data\processed\CIC-IDS2017_unified.csv.gz
Carga completada en 25.9 segundos.
Filas: 2,830,743
Columnas: 79
Uso de memoria: 1833.79 MB


## 3. Paso 1: Limpieza de datos

Se eliminan las filas que contienen valores `NaN`, `Infinity` y valores negativos en características temporales o de tasa cuya interpretación física no es válida.

In [5]:
# ============================================================
# Paso 1a: Identificación y eliminación de valores NaN e infinitos
# ============================================================

rows_before = len(df)
cols_before = len(df.columns)

# Conteo de NaN
n_nulls = df.isnull().sum().sum()

# Conteo de infinitos (solo en columnas numéricas)
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
n_infs = np.isinf(df[numeric_cols]).sum().sum()

print(f"Valores NaN detectados: {n_nulls:,}")
print(f"Valores infinitos detectados: {n_infs:,}")

# Reemplazar infinitos por NaN para poder eliminarlos juntos
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Eliminar filas con NaN
df = df.dropna().reset_index(drop=True)

rows_after = len(df)
log_step(
    step="1a. Eliminación de NaN e infinitos",
    rows_before=rows_before,
    rows_after=rows_after,
    cols_before=cols_before,
    cols_after=cols_before,
    detail=f"Se eliminaron {n_nulls:,} NaN y {n_infs:,} infinitos.",
)

Valores NaN detectados: 1,358
Valores infinitos detectados: 4,376
[1a. Eliminación de NaN e infinitos] Filas: 2,830,743 -> 2,827,876 (-2,867) | Columnas: 79 -> 79 (-0)
  Detalle: Se eliminaron 1,358 NaN y 4,376 infinitos.


In [6]:
# ============================================================
# Paso 1b: Eliminación de valores negativos inválidos
# ============================================================

rows_before = len(df)
cols_before = len(df.columns)

# Verificar cuáles columnas de la lista existen en el dataframe
existing_neg_cols = [c for c in TEMPORAL_NEGATIVE_COLS if c in df.columns]
missing_neg_cols = [c for c in TEMPORAL_NEGATIVE_COLS if c not in df.columns]

if missing_neg_cols:
    print(f"Advertencia: columnas no encontradas: {missing_neg_cols}")

# Contar negativos por columna antes de eliminar
neg_counts = (df[existing_neg_cols] < 0).sum()
print("Valores negativos por columna:")
display(neg_counts[neg_counts > 0].sort_values(ascending=False))

# Crear máscara de filas con valores negativos inválidos
neg_mask = (df[existing_neg_cols] < 0).any(axis=1)
n_neg_rows = neg_mask.sum()

# Eliminar filas con negativos
df = df[~neg_mask].reset_index(drop=True)

rows_after = len(df)
log_step(
    step="1b. Eliminación de valores negativos inválidos",
    rows_before=rows_before,
    rows_after=rows_after,
    cols_before=cols_before,
    cols_after=cols_before,
    detail=f"Se eliminaron {n_neg_rows:,} filas con valores negativos en columnas temporales/de tasa.",
)

Valores negativos por columna:


Flow IAT Min      2890
Flow IAT Mean      115
Flow Duration      115
Flow IAT Max       115
Flow Packets/s     115
Flow Bytes/s        85
Fwd IAT Min         17
dtype: int64

[1b. Eliminación de valores negativos inválidos] Filas: 2,827,876 -> 2,824,986 (-2,890) | Columnas: 79 -> 79 (-0)
  Detalle: Se eliminaron 2,890 filas con valores negativos en columnas temporales/de tasa.


## 4. Paso 2: Eliminación de columnas con varianza cero

Conforme al hallazgo del EDA, se eliminan las 8 columnas que mantienen el mismo valor en todos los registros y no aportan información discriminante.

In [7]:
# ============================================================
# Paso 2: Eliminación de columnas con varianza cero
# ============================================================

rows_before = len(df)
cols_before = len(df.columns)

# Verificar cuáles columnas existen realmente
existing_zero_cols = [c for c in ZERO_VAR_COLS if c in df.columns]
missing_zero_cols = [c for c in ZERO_VAR_COLS if c not in df.columns]

if missing_zero_cols:
    print(f"Advertencia: columnas no encontradas: {missing_zero_cols}")

# Verificar que efectivamente tienen varianza cero
var_check = df[existing_zero_cols].nunique()
print("Valores únicos por columna (debe ser 1):")
display(var_check)

# Eliminar columnas
df = df.drop(columns=existing_zero_cols)

rows_after = len(df)
cols_after = len(df.columns)
log_step(
    step="2. Eliminación de columnas con varianza cero",
    rows_before=rows_before,
    rows_after=rows_after,
    cols_before=cols_before,
    cols_after=cols_after,
    detail=f"Se eliminaron {len(existing_zero_cols)} columnas constantes.",
)

Valores únicos por columna (debe ser 1):


Bwd PSH Flags           1
Bwd URG Flags           1
Fwd Avg Bytes/Bulk      1
Fwd Avg Packets/Bulk    1
Fwd Avg Bulk Rate       1
Bwd Avg Bytes/Bulk      1
Bwd Avg Packets/Bulk    1
Bwd Avg Bulk Rate       1
dtype: int64

[2. Eliminación de columnas con varianza cero] Filas: 2,824,986 -> 2,824,986 (-0) | Columnas: 79 -> 71 (-8)
  Detalle: Se eliminaron 8 columnas constantes.


## 5. Paso 3: Codificación binaria de la variable objetivo

La columna `Label` se transforma a una variable binaria donde 0 representa tráfico benigno y 1 representa cualquier clase de ataque.

In [8]:
# ============================================================
# Paso 3: Codificación binaria de Label
# ============================================================

print("Distribución original de etiquetas:")
display(df["Label"].value_counts())

# Codificación binaria: BENIGN=0, cualquier ataque=1
df["Label_binary"] = (df["Label"].str.strip() != "BENIGN").astype(int)

print("\nDistribución binaria:")
display(df["Label_binary"].value_counts())

n_benign = (df["Label_binary"] == 0).sum()
n_attack = (df["Label_binary"] == 1).sum()
pct_benign = n_benign / len(df) * 100
pct_attack = n_attack / len(df) * 100

print(f"\nBENIGN (0): {n_benign:,} ({pct_benign:.2f}%)")
print(f"Ataque (1): {n_attack:,} ({pct_attack:.2f}%)")
print(f"Registros totales: {len(df):,}")

Distribución original de etiquetas:


Label
BENIGN                        2268624
DoS Hulk                       229965
PortScan                       158804
DDoS                           128006
DoS GoldenEye                   10288
FTP-Patator                      7931
SSH-Patator                      5895
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1956
Web Attack - Brute Force         1507
Web Attack - XSS                  652
Infiltration                       35
Web Attack - Sql Injection         21
Heartbleed                          7
Name: count, dtype: int64


Distribución binaria:


Label_binary
0    2268624
1     556362
Name: count, dtype: int64


BENIGN (0): 2,268,624 (80.31%)
Ataque (1): 556,362 (19.69%)
Registros totales: 2,824,986


## 6. Paso 4: Descarte semántico de columnas no numéricas

Se descartan las columnas que constituyen identificadores o metadatos de captura y no características de comportamiento del flujo. Se conserva exclusivamente la etiqueta binaria.

In [9]:
# ============================================================
# Paso 4: Descarte semántico
# ============================================================

rows_before = len(df)
cols_before = len(df.columns)

# Identificar columnas no numéricas
non_numeric_cols = df.select_dtypes(exclude=np.number).columns.tolist()
print(f"Columnas no numéricas detectadas: {non_numeric_cols}")

# Eliminar todas las columnas no numéricas (incluye Label original)
df = df.drop(columns=non_numeric_cols)

# Verificar que solo queden columnas numéricas + Label_binary
remaining_cols = df.columns.tolist()
print(f"\nColumnas restantes: {len(remaining_cols)}")
print(f"Última columna (debe ser Label_binary): {remaining_cols[-1]}")

# Separar características y etiqueta
feature_cols = [c for c in remaining_cols if c != "Label_binary"]
label_col = "Label_binary"

print(f"Características: {len(feature_cols)}")
print(f"Etiqueta: {label_col}")

cols_after = len(df.columns)
log_step(
    step="4. Descarte semántico de columnas no numéricas",
    rows_before=rows_before,
    rows_after=rows_before,
    cols_before=cols_before,
    cols_after=cols_after,
    detail=f"Se eliminaron {len(non_numeric_cols)} columnas: {non_numeric_cols}",
)

Columnas no numéricas detectadas: ['Label']

Columnas restantes: 71
Última columna (debe ser Label_binary): Label_binary
Características: 70
Etiqueta: Label_binary
[4. Descarte semántico de columnas no numéricas] Filas: 2,824,986 -> 2,824,986 (-0) | Columnas: 72 -> 71 (-1)
  Detalle: Se eliminaron 1 columnas: ['Label']


## 7. Paso 5: Partición estratificada Train/Test

Se divide el dataset mediante un Train-Test Split estratificado con semilla fija. Esta partición se realiza **antes** del filtro de correlación y la normalización para evitar fuga de datos hacia el conjunto de prueba.

In [10]:
# ============================================================
# Paso 5: Partición estratificada
# ============================================================

X = df[feature_cols]
y = df[label_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Distribución después de la partición:")
print(f"  Train: {len(X_train):,} registros")
print(f"    Benignos: {(y_train == 0).sum():,} ({(y_train == 0).mean() * 100:.2f}%)")
print(f"    Ataques:  {(y_train == 1).sum():,} ({(y_train == 1).mean() * 100:.2f}%)")
print(f"  Test:  {len(X_test):,} registros")
print(f"    Benignos: {(y_test == 0).sum():,} ({(y_test == 0).mean() * 100:.2f}%)")
print(f"    Ataques:  {(y_test == 1).sum():,} ({(y_test == 1).mean() * 100:.2f}%)")
print(f"\nLa estratificación mantiene la proporción de clases constante.")

Distribución después de la partición:
  Train: 2,259,988 registros
    Benignos: 1,814,899 (80.31%)
    Ataques:  445,089 (19.69%)
  Test:  564,998 registros
    Benignos: 453,725 (80.31%)
    Ataques:  111,273 (19.69%)

La estratificación mantiene la proporción de clases constante.


## 8. Paso 6: Filtro por correlación de Pearson

Se calcula la matriz de correlación **exclusivamente sobre el conjunto de entrenamiento** y se elimina una característica de cada par con correlación absoluta superior a 0.95. Esto evita la fuga de datos hacia el conjunto de prueba.

In [11]:
# ============================================================
# Paso 6: Filtro por correlación (solo en train)
# ============================================================

t0 = time.time()

# Calcular matriz de correlación SOLO sobre el conjunto de entrenamiento
corr_matrix = X_train.corr(method="pearson")

elapsed = time.time() - t0
print(f"Matriz de correlación calculada en {elapsed:.1f} segundos.")
print(f"Dimensiones: {corr_matrix.shape}")

# Identificar pares con correlación superior al umbral
high_corr_pairs = []
cols_to_remove = set()

columns = corr_matrix.columns.tolist()
for i in range(len(columns)):
    for j in range(i):
        r = corr_matrix.iloc[i, j]
        if pd.notna(r) and abs(r) > CORR_THRESHOLD:
            high_corr_pairs.append({
                "feature_1": columns[i],
                "feature_2": columns[j],
                "correlation": round(r, 6),
            })
            # Eliminar la segunda característica del par
            cols_to_remove.add(columns[i])

print(f"\nPares con correlación |r| > {CORR_THRESHOLD}: {len(high_corr_pairs)}")
print(f"Características a eliminar: {len(cols_to_remove)}")
print(f"Características retenidas: {len(feature_cols) - len(cols_to_remove)}")

# Guardar tabla de pares para el documento
df_high_corr = pd.DataFrame(high_corr_pairs).sort_values(
    "correlation", key=abs, ascending=False
)
df_high_corr.to_csv(TABLES_DIR / "02_pares_correlacion_train.csv", index=False)
print(f"\nTabla guardada en: {TABLES_DIR / '02_pares_correlacion_train.csv'}")

Matriz de correlación calculada en 17.4 segundos.
Dimensiones: (70, 70)

Pares con correlación |r| > 0.95: 45
Características a eliminar: 24
Características retenidas: 46

Tabla guardada en: C:\Proyectos\TT-deteccionIntrusiones\tables\02_pares_correlacion_train.csv


In [12]:
# ============================================================
# Aplicación del filtro de correlación
# ============================================================

cols_before = X_train.shape[1]

# Conservar solo las características no redundantes
selected_features = [c for c in feature_cols if c not in cols_to_remove]

X_train_filtered = X_train[selected_features].copy()
X_test_filtered = X_test[selected_features].copy()

cols_after = X_train_filtered.shape[1]
print(f"Características antes del filtro: {cols_before}")
print(f"Características después del filtro: {cols_after}")
print(f"Características eliminadas: {cols_before - cols_after}")

log_step(
    step="6. Filtro por correlación de Pearson",
    rows_before=len(X_train),
    rows_after=len(X_train_filtered),
    cols_before=cols_before,
    cols_after=cols_after,
    detail=f"Se eliminaron {len(cols_to_remove)} características con correlación > {CORR_THRESHOLD}.",
)

Características antes del filtro: 70
Características después del filtro: 46
Características eliminadas: 24
[6. Filtro por correlación de Pearson] Filas: 2,259,988 -> 2,259,988 (-0) | Columnas: 70 -> 46 (-24)
  Detalle: Se eliminaron 24 características con correlación > 0.95.


## 9. Paso 7: Normalización con StandardScaler

Se aplica estandarización ($\mu = 0$, $\sigma = 1$) a todas las características numéricas. El escalador se ajusta **exclusivamente sobre el conjunto de entrenamiento** y se aplica tanto a entrenamiento como a prueba, evitando así la fuga de datos.

In [13]:
# ============================================================
# Paso 7: Normalización con StandardScaler
# ============================================================

scaler = StandardScaler()

# Ajustar SOLO en train
X_train_scaled = scaler.fit_transform(X_train_filtered)

# Aplicar a test con los parámetros aprendidos de train
X_test_scaled = scaler.transform(X_test_filtered)

# Convertir de nuevo a DataFrame para facilitar su uso
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=selected_features,
    index=X_train_filtered.index,
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=selected_features,
    index=X_test_filtered.index,
)

print("Estandarización completada.")
print(f"Media de train (primeras 5 características): {X_train_scaled.iloc[:, :5].mean().round(6).tolist()}")
print(f"Desviación estándar de train (primeras 5): {X_train_scaled.iloc[:, :5].std().round(6).tolist()}")

log_step(
    step="7. Normalización con StandardScaler",
    rows_before=len(X_train_filtered),
    rows_after=len(X_train_scaled),
    cols_before=cols_after,
    cols_after=cols_after,
    detail="Escalador ajustado exclusivamente sobre el conjunto de entrenamiento.",
)

Estandarización completada.
Media de train (primeras 5 características): [0.0, -0.0, -0.0, -0.0, 0.0]
Desviación estándar de train (primeras 5): [1.0, 1.0, 1.0, 1.0, 1.0]
[7. Normalización con StandardScaler] Filas: 2,259,988 -> 2,259,988 (-0) | Columnas: 46 -> 46 (-0)
  Detalle: Escalador ajustado exclusivamente sobre el conjunto de entrenamiento.


## 10. Paso 8: Ponderación de clases

Conforme a la Sección 3.4.5, se calculan los pesos de clase inversamente proporcionales a su frecuencia, para compensar el desbalance sin alterar la distribución original de los datos.

In [14]:
# ============================================================
# Paso 8: Cálculo de ponderación de clases
# ============================================================

# Calcular pesos a partir de las etiquetas de entrenamiento
classes = np.array([0, 1])
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train.values,
)

class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print("Pesos de clase calculados (inversamente proporcionales a la frecuencia):")
print(f"  Clase 0 (BENIGN): {class_weight_dict[0]:.4f}")
print(f"  Clase 1 (Ataque): {class_weight_dict[1]:.4f}")
print(f"  Ratio de ponderación (ataque/benigno): {class_weight_dict[1] / class_weight_dict[0]:.2f}")

# Guardar para uso en entrenamiento
class_weight_path = PROCESSED_DIR / "class_weights.pkl"
with open(class_weight_path, "wb") as f:
    pickle.dump(class_weight_dict, f)
print(f"\nPesos guardados en: {class_weight_path}")

Pesos de clase calculados (inversamente proporcionales a la frecuencia):
  Clase 0 (BENIGN): 0.6226
  Clase 1 (Ataque): 2.5388
  Ratio de ponderación (ataque/benigno): 4.08

Pesos guardados en: C:\Proyectos\TT-deteccionIntrusiones\data\processed\class_weights.pkl


## 11. Paso 9: Transformación secuencial para LSTM

Conforme a la Sección 3.4.6, se aplica la técnica de ventana deslizante sobre los datos ya preprocesados para generar los tensores tridimensionales requeridos por la arquitectura LSTM. La ventana se construye **dentro de cada partición** (train y test) para evitar fuga de datos entre ellas.

In [15]:
# ============================================================
# Paso 9: Transformación secuencial (ventana deslizante)
# ============================================================

def create_sequences(X: np.ndarray, y: np.ndarray, window_size: int):
    """
    Genera secuencias temporales mediante ventana deslizante.

    Args:
        X: Matriz de características (n_muestras, n_caracteristicas).
        y: Vector de etiquetas binarias.
        window_size: Número de pasos de tiempo por secuencia.

    Returns:
        X_seq: Tensor tridimensional (n_secuencias, window_size, n_caracteristicas).
        y_seq: Vector de etiquetas para cada secuencia (se toma la etiqueta
               del último paso de la ventana).
    """
    X_seq, y_seq = [], []
    for i in range(len(X) - window_size + 1):
        X_seq.append(X[i:i + window_size])
        y_seq.append(y[i + window_size - 1])
    return np.array(X_seq), np.array(y_seq)


print(f"Generando secuencias con ventana de {WINDOW_SIZE} pasos...")
t0 = time.time()

# Generar secuencias por partición (sin mezcla entre train y test)
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled.values, y_train.values, WINDOW_SIZE
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled.values, y_test.values, WINDOW_SIZE
)

elapsed = time.time() - t0

print(f"\nSecuencias generadas en {elapsed:.1f} segundos.")
print(f"  Train: {X_train_seq.shape} -> {X_train_seq.shape[0]:,} secuencias")
print(f"  Test:  {X_test_seq.shape} -> {X_test_seq.shape[0]:,} secuencias")
print(f"\nDistribución de etiquetas en secuencias:")
print(f"  Train benignos: {(y_train_seq == 0).sum():,} | ataques: {(y_train_seq == 1).sum():,}")
print(f"  Test  benignos: {(y_test_seq == 0).sum():,} | ataques: {(y_test_seq == 1).sum():,}")

Generando secuencias con ventana de 10 pasos...

Secuencias generadas en 17.3 segundos.
  Train: (2259979, 10, 46) -> 2,259,979 secuencias
  Test:  (564989, 10, 46) -> 564,989 secuencias

Distribución de etiquetas en secuencias:
  Train benignos: 1,814,892 | ataques: 445,087
  Test  benignos: 453,718 | ataques: 111,271


In [16]:
# ============================================================
# Guardado de secuencias LSTM en formato NumPy comprimido
# ============================================================

seq_dir = PROCESSED_DIR / "lstm_sequences"
seq_dir.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    seq_dir / "lstm_sequences.npz",
    X_train_seq=X_train_seq,
    y_train_seq=y_train_seq,
    X_test_seq=X_test_seq,
    y_test_seq=y_test_seq,
    window_size=WINDOW_SIZE,
)

seq_file_size = (seq_dir / "lstm_sequences.npz").stat().st_size / (1024**2)
print(f"Secuencias guardadas en: {seq_dir / 'lstm_sequences.npz'}")
print(f"Tamaño del archivo: {seq_file_size:.2f} MB")

Secuencias guardadas en: C:\Proyectos\TT-deteccionIntrusiones\data\processed\lstm_sequences\lstm_sequences.npz
Tamaño del archivo: 501.00 MB


## 12. Guardado de artefactos del pipeline

Se guardan todos los objetos necesarios para la fase de entrenamiento: datos tabulares, escalador, lista de características seleccionadas y registro del pipeline.

In [17]:
# ============================================================
# Guardado de datos tabulares para Random Forest y SVM
# ============================================================

# Datos de entrenamiento y prueba (tabulares, ya normalizados)
tabular_path = PROCESSED_DIR / "tabular_data.npz"
np.savez_compressed(
    tabular_path,
    X_train=X_train_scaled.values,
    y_train=y_train.values,
    X_test=X_test_scaled.values,
    y_test=y_test.values,
    feature_names=np.array(selected_features),
)

tab_size = tabular_path.stat().st_size / (1024**2)
print(f"Datos tabulares guardados en: {tabular_path}")
print(f"Tamaño: {tab_size:.2f} MB")

# Escalador (necesario para inferencia futura)
scaler_path = PROCESSED_DIR / "scaler.pkl"
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
print(f"Escalador guardado en: {scaler_path}")

# Lista de características seleccionadas
features_path = PROCESSED_DIR / "selected_features.pkl"
with open(features_path, "wb") as f:
    pickle.dump(selected_features, f)
print(f"Lista de características guardada en: {features_path}")

# Etiquetas originales para análisis posterior por familia de ataque
labels_path = PROCESSED_DIR / "labels_original_train_test.pkl"
# Recuperar las etiquetas originales del dataframe antes de la codificación
# Se necesitan los índices de train y test
train_idx = X_train.index
test_idx = X_test.index

labels_dict = {
    "train_idx": train_idx.values,
    "test_idx": test_idx.values,
}
with open(labels_path, "wb") as f:
    pickle.dump(labels_dict, f)
print(f"Índices de partición guardados en: {labels_path}")

Datos tabulares guardados en: C:\Proyectos\TT-deteccionIntrusiones\data\processed\tabular_data.npz
Tamaño: 256.22 MB
Escalador guardado en: C:\Proyectos\TT-deteccionIntrusiones\data\processed\scaler.pkl
Lista de características guardada en: C:\Proyectos\TT-deteccionIntrusiones\data\processed\selected_features.pkl
Índices de partición guardados en: C:\Proyectos\TT-deteccionIntrusiones\data\processed\labels_original_train_test.pkl


In [20]:
# ============================================================
# Registro del pipeline
# ============================================================

df_pipeline_log = pd.DataFrame(pipeline_log)
log_path = LOGS_DIR / "02_pipeline_log.csv"
df_pipeline_log.to_csv(log_path, index=False)

print("Registro del pipeline:")
print("=" * 80)
display(df_pipeline_log)
print(f"\nLog guardado en: {log_path}")

Registro del pipeline:


,step,rows_before,rows_after,rows_removed,cols_before,cols_after,cols_removed,detail
0,1a. Eliminación de NaN e infinitos,2830743,2827876,2867,79,79,0,"Se eliminaron 1,358 NaN y 4,376 infinitos."
1,1b. Eliminación de valores negativos inválidos,2827876,2824986,2890,79,79,0,"Se eliminaron 2,890 filas con valores negativo..."
2,2. Eliminación de columnas con varianza cero,2824986,2824986,0,79,71,8,Se eliminaron 8 columnas constantes.
3,4. Descarte semántico de columnas no numéricas,2824986,2824986,0,72,71,1,Se eliminaron 1 columnas: ['Label']
4,6. Filtro por correlación de Pearson,2259988,2259988,0,70,46,24,Se eliminaron 24 características con correlaci...
5,7. Normalización con StandardScaler,2259988,2259988,0,46,46,0,Escalador ajustado exclusivamente sobre el con...



Log guardado en: C:\Proyectos\TT-deteccionIntrusiones\logs\02_pipeline_log.csv


## 13. Resumen del preprocesamiento

In [21]:
# ============================================================
# Resumen final del preprocesamiento
# ============================================================

summary = {
    "Indicador": [
        "Registros iniciales (dataset unificado)",
        "Registros después de limpieza",
        "Filas eliminadas (NaN + Inf + negativos)",
        "Columnas iniciales",
        "Columnas de varianza cero eliminadas",
        "Columnas eliminadas por correlación",
        "Características finales",
        "Registros de entrenamiento",
        "Registros de prueba",
        "Proporción benigno/ataque en train",
        "Peso clase benigna (0)",
        "Peso clase ataque (1)",
        "Tamaño de ventana LSTM",
        "Secuencias de entrenamiento (LSTM)",
        "Secuencias de prueba (LSTM)",
        "Dimensión del tensor LSTM (train)",
        "Semilla aleatoria",
    ],
    "Valor": [
        f"{2830743:,}",
        f"{len(df):,}",
        f"{2830743 - len(df):,}",
        "79",
        f"{len(existing_zero_cols)}",
        f"{len(cols_to_remove)}",
        f"{len(selected_features)}",
        f"{len(X_train):,}",
        f"{len(X_test):,}",
        f"{(y_train == 0).mean() * 100:.2f}% / {(y_train == 1).mean() * 100:.2f}%",
        f"{class_weight_dict[0]:.4f}",
        f"{class_weight_dict[1]:.4f}",
        f"{WINDOW_SIZE}",
        f"{X_train_seq.shape[0]:,}",
        f"{X_test_seq.shape[0]:,}",
        str(X_train_seq.shape),
        str(RANDOM_STATE),
    ],
}

df_summary = pd.DataFrame(summary)
summary_path = TABLES_DIR / "02_resumen_preprocesamiento.csv"
df_summary.to_csv(summary_path, index=False)

print("Resumen final del preprocesamiento")
print("=" * 80)
display(df_summary)
print(f"\nTabla guardada en: {summary_path}")

Resumen final del preprocesamiento


,Indicador,Valor
0,Registros iniciales (dataset unificado),"2,830,743"
1,Registros después de limpieza,"2,824,986"
2,Filas eliminadas (NaN + Inf + negativos),"5,757"
3,Columnas iniciales,79
4,Columnas de varianza cero eliminadas,8
5,Columnas eliminadas por correlación,24
6,Características finales,46
7,Registros de entrenamiento,"2,259,988"
8,Registros de prueba,"564,998"
9,Proporción benigno/ataque en train,80.31% / 19.69%



Tabla guardada en: C:\Proyectos\TT-deteccionIntrusiones\tables\02_resumen_preprocesamiento.csv


## 14. Archivos generados

Al finalizar la ejecución de este notebook, se generan los siguientes artefactos en `data/processed/`:

| Archivo | Contenido | Uso |
|---------|-----------|-----|
| `tabular_data.npz` | X_train, y_train, X_test, y_test normalizados | Random Forest y SVM |
| `lstm_sequences/lstm_sequences.npz` | Tensores 3D con ventana deslizante | LSTM |
| `scaler.pkl` | StandardScaler ajustado | Reproducibilidad |
| `selected_features.pkl` | Lista de características finales | Documentación |
| `class_weights.pkl` | Pesos de clase | Entrenamiento de modelos |

Y en `tables/`:

| Archivo | Contenido |
|---------|-----------|
| `02_pipeline_log.csv` | Registro paso a paso del pipeline |
| `02_pares_correlacion_train.csv` | Pares eliminados por correlación |
| `02_resumen_preprocesamiento.csv` | Tabla resumen de resultados |